# Micrograd v2 Practice

Goal: implement a minimal `Value` class from memory, then verify gradients.

Rules:
- Try without looking at yesterday's notebook first.
- If stuck for 20-30 minutes, check notes briefly, then close them again.
- Record mistakes in the notes section at the bottom.

## Task 1: Implement `Value`

Required API:

```python
class Value:
    def __init__(self, data, _children=(), _op='', label=''):
        ...

    def __repr__(self):
        ...

    def __add__(self, other):
        ...

    def __mul__(self, other):
        ...

    def tanh(self):
        ...

    def backward(self):
        ...
```

Make this pass:

```python
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
L = (a * b + c).tanh()
L.backward()

L.data, a.grad, b.grad, c.grad
```

In [113]:
# Write your Value class here. Try again from memory.
# Mistakes from attempt 1 are listed at the bottom.

import math

class Value:
    def __init__(self, data, _children=(), _op='', label=''):

        self.data = data

        self._prev = set(_children)

        self.label = label

        self.grad = 0.0

        self._backward = lambda : None
        

    def __repr__(self):

        return f"Value(data = {self.data}, label = '{self.label}')"
        

    def __add__(self, other):

        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data + other.data, (self, other), '+')

        def _backward():

            self.grad += out.grad

            other.grad += out.grad
            
        out._backward = _backward

        return out
        

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        out = Value(self.data * other.data, (self, other), '*')

        def _backward():

            self.grad += other.data * out.grad

            other.grad += self.data * out.grad

        out._backward = _backward

        return out
        

    def tanh(self):

        x = self.data

        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)

        out = Value(t, (self,), 'tanh')

        def _backward():

            self.grad += (1 - t**2) * out.grad

        out._backward = _backward

        return out


    def __neg__(self):
        return self * -1
    

    def __sub__(self, other):
        return self + (-other)
        
    
    def __rsub__(self, other):
        return other + (-self)

    def __radd__(self, other):
        return self + other

        
    
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
    
        def _backward():
            self.grad += other * self.data**(other - 1) * out.grad
    
        out._backward = _backward
        return out
        


    def backward(self):

        topo = []

        visited = set()

        def build_topo(v):

            if v not in visited:

                visited.add(v)

                for child in v._prev:

                    build_topo(child)

                topo.append(v)


        build_topo(self)

        self.grad = 1.0
        


        for node in reversed(topo):

            node._backward()

            
        

            


## Task 2: Gradient Checks

Run these after implementing `Value`.

In [65]:
# Check 1: main expression
a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
L = (a * b + c).tanh()
L.backward()

print('L.data =', L.data)
print('a.grad =', a.grad)
print('b.grad =', b.grad)
print('c.grad =', c.grad)


L.data = 0.9999997749296758
a.grad = -1.3504217930560003e-06
b.grad = 9.002811953706669e-07
c.grad = 4.5014059768533343e-07


In [64]:
# Check 2: reused node through addition
a = Value(2.0, label='a')
L = a + a
L.backward()
print('expected a.grad = 2.0')
print('actual a.grad =', a.grad)


expected a.grad = 2.0
actual a.grad = 2.0


In [63]:
# Check 3: reused node through multiplication
a = Value(3.0, label='a')
L = a * a
L.backward()
print('expected a.grad = 6.0')
print('actual a.grad =', a.grad)


expected a.grad = 6.0
actual a.grad = 6.0


## Mistake Notes

### Attempt 1

- `__init__`: remember to initialize `self.grad = 0.0`. Every `Value` needs a gradient slot before backward can accumulate into it.
- `__init__`: use the standard names `_children` and `self._prev = set(_children)`. The graph traversal in `backward()` expects `v._prev`.
- `__init__`: initialize `self._backward = lambda: None` so leaf nodes have a harmless default backward function.
- `__add__`, `__mul__`, `tanh`: return `out`. Otherwise expressions like `a * b + c` receive `None`.
- `__mul__` backward: use `other.data` and `self.data`, not `other` and `self`, because the local derivative needs scalar values.
- `tanh`: store the operation label as `'tanh'` and return `out`.
- `backward`: build a topological order first, then iterate through `reversed(topo)` so gradients flow from final output back to inputs.

Skeleton for `backward()` idea:

```python
topo = []
visited = set()

def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(self)
self.grad = 1.0
for node in reversed(topo):
    node._backward()
```


### Attempt 2

- Typo: wrote `sefl._backward`; should be `self._backward`. Attribute spelling must be exact.
- `__repr__`: use `{self.data}` and `{self.label}` inside the f-string. Otherwise Python prints literal text instead of attribute values.
- `__add__` / `__mul__`: remember to `return out`. Without returning, chained expressions receive `None`.
- `__mul__` backward: use `+=`, not `=`, because reused nodes like `a * a` need gradient contributions to accumulate.
- `__mul__` backward: use scalar values `other.data` and `self.data`, not the `Value` objects themselves.
- `tanh`: `x` should be the scalar `self.data`, not a new `Value`. Compute scalar `t`, then wrap it with `out = Value(t, (self,), 'tanh')`.
- `tanh`: formula should be `(math.exp(2*x) - 1) / (math.exp(2*x) + 1)`. Parentheses matter.
- `tanh`: define `out` before assigning `out._backward`, and return `out`.
- `backward`: inside `build_topo`, append the current node after recursively visiting children: `topo.append(v)`.
- `backward`: call `build_topo(self)` before setting the seed gradient.
- `backward`: set `self.grad = 1.0`, not `out.grad`, because `self` is the final node that called `backward()`.
- `backward`: final loop should call `node._backward()`. Do not assign to `node._backward`.

Third-pass focus:

- `self._backward = lambda: None`
- every operation creates `out`, stores local `_backward`, then `return out`
- local gradient contributions use `+=`
- `backward()` = `topo + visited`, recursive `build_topo`, `build_topo(self)`, `self.grad = 1.0`, `reversed(topo)`


### Attempt 3

- `__mul__` backward is now correct with `+=`; this fixed the reused-node case `a * a`, where `a.grad` should accumulate two paths and become `6.0` when `a = 3.0`.
- `tanh`: the formula must use `2*x`, not `x**2`: `(math.exp(2*x) - 1) / (math.exp(2*x) + 1)`.
- `tanh` backward should also use `+=`: `self.grad += (1 - t**2) * out.grad`, because the input node may receive gradients from multiple paths.
- Jupyter workflow: after editing the `Value` class, rerun the class cell before rerunning the check cells. Otherwise the kernel may still be using the old class definition.
- Debug checklist: if Check 2 and Check 3 pass but Check 1 looks numerically strange, inspect operation-specific math like `tanh`, not the whole `backward()` structure.


### MLP Toy Demo Mistakes

- Random initialization: use `random.uniform(-1, 1)`, not `random(-1, 1)`. `random` is the module; `uniform` is the function.
- Creating weights: use `range(nin)`, not `for _ in nin`, because `nin` is an integer input count.
- Neuron forward pass: use `sum((wi * xi for wi, xi in zip(self.w, x)), self.b)`. The bias is the `start` value of `sum`, not something added into `zip`.
- Layer forward pass: call every neuron on the same input `x`: `outs = [n(x) for n in self.neurons]`.
- MLP forward pass: `return x` must be outside the `for layer in self.layers` loop, otherwise the model returns after only the first layer.
- Model creation: `n = MLP(3, [4, 4, 1])`, not `n = [3, [4, 4, 1]]`. The first creates a callable model object; the second is just a list.
- Backward call: use `loss.backward()`. Without parentheses, `loss.backward` only references the function and does not run backprop.
- Parameter update: `p.data` is one scalar parameter value. `n.parameters()` is the list of all parameters; the loop updates each scalar one at a time.
- `sum` and `Value`: Python `sum` starts from integer `0` by default. Either implement `__radd__` or start from `Value(0.0)` when summing `Value` objects.


## Recall Summary Before MLP

Use this as the mental model before writing the MLP code.

- `topo` stores `Value` node objects in input-to-output order. Each object carries `data`, `grad`, `_prev`, and `_backward`.
- `topo.append(v)` adds the current node `v` into the `topo` list after its children have been visited.
- `reversed(topo)` lets gradients flow from the final output back toward inputs.
- `node._backward()` uses the current node's existing `node.grad` and local derivative rules to pass gradient contributions to previous nodes in `_prev`.
- Use `+=` because a reused node can receive gradient contributions through multiple graph paths.


## Task 3: MLP Toy Demo Recall

Fill the blanks in order. Do not look at the original code unless you are stuck for more than 10 minutes.

Goal: implement `Neuron`, `Layer`, `MLP`, collect `parameters()`, then run a tiny training loop where loss decreases.


In [89]:
# Step 1: Neuron
# Rewrite from memory after defining the responsibility in words.

import random
import math

class Neuron:
    def __init__(self, nin):
        # TODO: create nin weights and one bias
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        # TODO: compute tanh(sum(wi * xi) + b)
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = act.tanh()
        return out
        

    def parameters(self):
        # TODO: return weights plus bias
        return self.w + [self.b]

# quick check
n = Neuron(3)
n([2.0, 3.0, -1.0])
# n.parameters()


Value(data = 0.9639110540936533, label = '')

In [91]:
# Step 2: Layer

class Layer:
    def __init__(self, nin, nout):
        # TODO: create nout neurons, each neuron has nin inputs
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        # TODO: call every neuron on the same input x
        # return one Value if there is only one neuron, otherwise return a list
        outs = [n(x) for n in self.neurons]

        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        # TODO: collect all parameters from all neurons
        return [p for neuron in self.neurons for p in neuron.parameters()]

# quick check
layer = Layer(3, 4)
layer([2.0, 3.0, -1.0])
len(layer.parameters())


16

In [117]:
# Step 3: MLP

class MLP:
    def __init__(self, nin, nouts):
        # TODO: build sizes list, e.g. nin=3, nouts=[4,4,1] -> [3,4,4,1]
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        # TODO: forward x through every layer
        for layer in self.layers:
            x = layer(x)
        return x
            

    def parameters(self):
        # TODO: collect all parameters from all layers
        return [p for layer in self.layers for p in layer.parameters()]

# quick check
n = MLP(3, [4, 4, 1])
n([2.0, 3.0, -1.0])
len(n.parameters())


41

In [162]:
# Step 4: training loop
# Fill this after Neuron / Layer / MLP work.

xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0]

# TODO: create model
n = MLP(3, [4, 4, 1])

for k in range(20):
    # TODO: forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))

    # TODO: zero grad
    for p in n.parameters():
        p.grad = 0
    

    # TODO: backward
    loss.backward()

    # TODO: update parameters
    for p in n.parameters():
        p.data += -0.1 * p.grad

    # TODO: print loss
    print(k, loss.data)


0 6.57880972840939
1 4.5936712620115925
2 4.081583082525386
3 4.542472563298127
4 5.036247093975737
5 5.361601461405389
6 1.6179292422263656
7 2.6199294759016243
8 0.7394917720241238
9 0.0931206004327767
10 0.07589111768677992
11 0.06488320372792954
12 0.057019835023848504
13 0.05102444556194369
14 0.04625461381959852
15 0.042344566028991754
16 0.03906732157946229
17 0.03627282968164044
18 0.03385699332983362
19 0.03174480967290434


## MLP Debug Checklist

- If `int + Value` errors inside `sum`, use `sum(generator, self.b)` so the start value is a `Value`.
- If reused nodes have wrong gradients, check for `+=` in operation backward functions.
- If loss becomes exactly `0.0` immediately, check whether `loss` accidentally used a stale value or whether the model was already trained in this kernel.
- If loss explodes, check learning rate, zero grad, and whether the update sign is `p.data += -lr * p.grad`.
- If changing code has no effect, rerun the class definition cells before rerunning checks.
